# Hybrid LNN + Wave + Analogical Memory on Colab

This notebook trains the hybrid LNN with both imported NEXUS parts enabled on the full-xTB path:
- wave / quantum bridge
- analogical memory bank

Run top-to-bottom.

In [ ]:
# Cell 1: clone/pull repo and install package
import os, subprocess

REPO = 'https://github.com/Nikku03/enzyme_Software.git'
REPO_DIR = '/content/enzyme_Software'

if not os.path.exists(REPO_DIR):
    print('Cloning repo...')
    subprocess.run(f'git clone {REPO} {REPO_DIR}', shell=True, check=True)
else:
    print('Repo exists, pulling latest...')
    subprocess.run(f'git -C {REPO_DIR} pull --ff-only', shell=True, check=True)

# Clear stale bytecode so updated source files (e.g. hybrid_model.py) are recompiled
subprocess.run(f'find {REPO_DIR}/src -name "*.pyc" -delete', shell=True)
subprocess.run(f'find {REPO_DIR}/src -name "__pycache__" -type d -exec rm -rf {{}} +', shell=True)

subprocess.run(f'pip -q install -e {REPO_DIR}', shell=True, check=True)
print('Repo ready:', REPO_DIR)


In [2]:
# Cell 2: mount Google Drive for persistent checkpoints/cache
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/enzyme_hybrid_lnn/checkpoints', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/enzyme_hybrid_lnn/cache/manual_engine_full', exist_ok=True)
    print('Drive mounted.')
except Exception as e:
    print('Drive mount skipped:', e)

Mounted at /content/drive
Drive mounted.


In [ ]:
# Cell 3: choose preset / overrides
import os

# ── Preset ────────────────────────────────────────────────────────────────────
os.environ['HYBRID_COLAB_PRESET'] = 'balanced'
os.environ['HYBRID_COLAB_LOCK_PRESET_POLICY'] = '1'

# ── Paths ─────────────────────────────────────────────────────────────────────
os.environ['HYBRID_COLAB_STRUCTURE_SDF'] = '3D structures.sdf'
os.environ['HYBRID_COLAB_OUTPUT_DIR'] = '/content/drive/MyDrive/enzyme_hybrid_lnn/checkpoints/hybrid_full_xtb'
os.environ['HYBRID_COLAB_ARTIFACT_DIR'] = '/content/drive/MyDrive/enzyme_hybrid_lnn/artifacts/hybrid_full_xtb'
os.environ['HYBRID_COLAB_MANUAL_CACHE_DIR'] = '/content/drive/MyDrive/enzyme_hybrid_lnn/cache/manual_engine_full'
os.environ['HYBRID_COLAB_XTB_CACHE_DIR'] = '/content/drive/MyDrive/enzyme_hybrid_lnn/cache/full_xtb'

# ── Warm-start choice ─────────────────────────────────────────────────────────
# Safe option: honest full-data checkpoint family
os.environ['HYBRID_COLAB_WARM_START'] = '/content/drive/MyDrive/enzyme_hybrid_lnn/checkpoints/hybrid_full_xtb/hybrid_full_xtb_best.pt'
os.environ.pop('HYBRID_COLAB_WARM_START_REPORT', None)
os.environ['HYBRID_COLAB_WARM_START_MODE'] = 'best'

# If you want to warm-start from a report instead, comment the line above and set:
# os.environ['HYBRID_COLAB_WARM_START_REPORT'] = '/content/drive/MyDrive/enzyme_hybrid_lnn/revisit/artifacts/attempt_08_arch.prior_biased_bridge/hybrid_full_xtb_report_20260401_024651.json'

# ── Optional runtime knobs ────────────────────────────────────────────────────
# Leave unset to keep preset defaults.
os.environ.pop('HYBRID_COLAB_PRECEDENT_LOGBOOK', None)

# ── Colab torch stability ─────────────────────────────────────────────────────
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['TORCH_COMPILE_DISABLE'] = '1'
os.environ['HYBRID_FORCE_MANUAL_OPTIMIZER'] = '1'

print('Configured overrides before wrapper exec:')
for key in [
    'HYBRID_COLAB_PRESET',
    'HYBRID_COLAB_LOCK_PRESET_POLICY',
    'HYBRID_COLAB_WARM_START',
    'HYBRID_COLAB_WARM_START_REPORT',
    'HYBRID_COLAB_WARM_START_MODE',
    'HYBRID_COLAB_OUTPUT_DIR',
    'HYBRID_COLAB_ARTIFACT_DIR',
]:
    value = os.environ.get(key)
    if value is not None:
        print(f'  {key}={value}')

print()
print('Balanced preset now resolves to:')
print('  dataset: data/prepared_training/main8_site_conservative_singlecyp_clean_symm.json')
print('  split_mode: scaffold_source_size')
print('  epochs: 60')
print('  lr: 2e-4')
print('  wd: 5e-4')
print('  batch_size: 16')
print('  early_stopping_metric: site_top1')
print('  early_stopping_patience: 8')
print('  backbone_freeze_epochs: 50')
print('  include_xenosite: 1')
print('  disable_precedent_logbook: 1')
print()
print('Expected effective split after filtering is roughly: train 790 / val 72 / test 110.')



In [7]:
# Cell 4: train hybrid LNN + both imported NEXUS parts
exec(open('/content/enzyme_Software/scripts/colab_train_hybrid_lnn.py').read(), {'__name__': '__main__'})



Hybrid LNN Colab wrapper
preset=balanced
output_dir=/content/drive/MyDrive/enzyme_hybrid_lnn/checkpoints/hybrid_full_xtb
artifact_dir=/content/drive/MyDrive/enzyme_hybrid_lnn/artifacts/hybrid_full_xtb
manual_cache_dir=/content/drive/MyDrive/enzyme_hybrid_lnn/cache/manual_engine_full
xtb_cache_dir=/content/drive/MyDrive/enzyme_hybrid_lnn/cache/full_xtb
warm_start=/content/drive/MyDrive/enzyme_hybrid_lnn/checkpoints/hybrid_full_xtb/hybrid_full_xtb_latest.pt
disable_precedent_logbook=1
TORCHDYNAMO_DISABLE=1
TORCH_COMPILE_DISABLE=1
HYBRID_FORCE_MANUAL_OPTIMIZER=1
HYBRID_COLAB_BATCH_SIZE=16
HYBRID_COLAB_COMPUTE_XTB_IF_MISSING=0
HYBRID_COLAB_DATASET=data/prepared_training/main5_site_conservative_singlecyp_clean.json
HYBRID_COLAB_DISABLE_PRECEDENT_LOGBOOK=1
HYBRID_COLAB_EARLY_STOPPING_PATIENCE=3
HYBRID_COLAB_EPOCHS=12
HYBRID_COLAB_FREEZE_NEXUS_MEMORY=1
HYBRID_COLAB_INCLUDE_XENOSITE=1
HYBRID_COLAB_LIMIT=0
HYBRID_COLAB_LR=5e-5
HYBRID_COLAB_SEED=42
HYBRID_COLAB_SITE_LABELED_ONLY=1
HYBRID_COLAB_S

In [5]:
import os, subprocess

REPO_DIR = '/content/enzyme_Software'

print(f'Installing dependencies from {REPO_DIR}/requirements.txt...')
subprocess.run(f'pip install -q -r {REPO_DIR}/requirements.txt', shell=True, check=True)

Installing dependencies from /content/enzyme_Software/requirements.txt...


CompletedProcess(args='pip install -q -r /content/enzyme_Software/requirements.txt', returncode=0)

In [ ]:
# Cell 5: optional single-molecule smoke prediction after training
import os, subprocess
SMILES = 'CCOc1ccc2nc(S(N)(=O)=O)sc2c1'
CKPT = '/content/drive/MyDrive/enzyme_hybrid_lnn/checkpoints/hybrid_full_xtb/hybrid_full_xtb_best.pt'
if os.path.exists(CKPT):
    cmd = [
        'python', '/content/enzyme_Software/scripts/run_hybrid_prediction.py',
        '--smiles', SMILES,
        '--checkpoint', CKPT,
    ]
    subprocess.run(cmd, check=True)
else:
    print('Checkpoint not found yet:', CKPT)